In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder

car_data = pd.read_csv("car_prices.csv", on_bad_lines='skip')
#print(car_data.shape)
#car_data.head()

y = car_data.sellingprice


features = ['year', 'condition', 'odometer', 'make']
X = car_data[features]

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)


train_X_curat = train_X.copy()
val_X_curat = val_X.copy()


# aici scap de missing rows:

my_imputer = SimpleImputer()
# imputed_X_train = pd.DataFrame(my_imputer.fit_transform(train_X[['year', 'condition', 'odometer']]))
# imputed_X_valid = pd.DataFrame(my_imputer.transform(val_X[['year', 'condition', 'odometer']]))

# imputed_X_train.columns = ['year', 'condition', 'odometer']
# imputed_X_valid.columns = ['year', 'condition', 'odometer']

train_X_curat[['year', 'condition', 'odometer']] = my_imputer.fit_transform(train_X[['year', 'condition', 'odometer']])
val_X_curat[['year', 'condition', 'odometer']] = my_imputer.transform(val_X[['year', 'condition', 'odometer']])

# adaug encoder pentru 'make' :

my_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

label_X_train = train_X_curat.copy()
label_X_valid = val_X_curat.copy()

label_X_train[['make']] = my_encoder.fit_transform(train_X_curat[['make']])
label_X_valid[['make']] = my_encoder.transform(val_X_curat[['make']])

train_X_curat[['make']] = label_X_train[['make']]
val_X_curat[['make']] = label_X_valid[['make']]

car_model = RandomForestRegressor(random_state = 0, n_jobs= -1) # n_jobs ii spune compilatorului ca poate folosi toate nucleele necesare, a scazut timpul de antrenat de la 1.5 minute la 23s
car_model.fit(train_X_curat, train_y)
prediction = car_model.predict(val_X_curat)
print(mean_absolute_error(val_y, prediction))

#car_data['make'].nunique() ====>> 96 de valori distince, one hot encoder mi ar bubui calculatorul => cardinal encoder

3743.04699455378
